## 0. Kernel setup (run in a terminal, not in this notebook)

Before launching this notebook, create and select a conda environment kernel (`2ndWorkshop`).

### Purdue Gilbreth cluster

```bash
module load conda
conda-env-mod create -n ENV_NAME_HERE -j
module use $HOME/privatemodules
module load conda-env/ENV_NAME_HERE-py3.10.11
```

Replace `ENV_NAME_HERE` with your environment name (`2ndWorkshop`), then select the matching kernel in Jupyter before running the cells below.

Check the env and kernel were created:
```bash
conda env list
jupyter kernelspec list
```

### Local machine (conda)

```bash
conda create -n 2ndWorkshop python=3.10 pip -y
conda activate 2ndWorkshop
pip install ipykernel
python -m ipykernel install --user --name 2ndWorkshop --display-name "Python (2ndWorkshop)"
```

Select the `Python (2ndWorkshop)` kernel, run the install cell below once, then restart the kernel.

**Prerequisite:** `Workshop2_Part2a_Server.ipynb` must already be running (in a separate
kernel) before you run the cells below.

**Note:** This notebook needs an LLM backend that supports tool calling — Ollama running
locally is the free default (`ollama pull llama3.2`), or set a Purdue GenAI / OpenAI key.
See the README for details.


# Workshop 2, Part 2b: LangGraph Agent (Boilermaker TA)

This is the **agent half** of Part 2. It connects to the MCP tool server from
`Workshop2_Part2a_Server.ipynb` over HTTP and orchestrates its tools in a fixed
**LangGraph** workflow — gather context, then write the answer — to complete a task.
That's the Boilermaker TA.

**Before running this notebook:** start `Workshop2_Part2a_Server.ipynb` in a separate
kernel and leave it running.

**What you'll do:**
- Connect to the MCP server and pick out the three tools this workflow calls by name
- Build a two-node LangGraph graph: gather context, then write
- Run the Boilermaker TA on a real task and inspect what each step produces

**LLM backend:** same default as Workshop 1 and Part 1 — a small local Hugging Face
model, no API key or external server required. Tool-calling reliability drops with model
size, though, so **C2** also shows how to switch to Ollama, OpenAI, or Anthropic with one
line if the local model struggles.


## 0. Install dependencies

Run once, then restart the kernel.

In [1]:
! pip install torch transformers accelerate langchain-huggingface langchain langchain-core langgraph langchain-mcp-adapters langchain-ollama python-dotenv


---
# Part C: LangGraph Agent

The agent is a **fixed-order workflow graph** with two nodes, run in a straight line:
- `gather_context` — calls both lookup tools (concurrently, via `asyncio.gather`) using
  only the original question
- `write` — runs after; the only node that calls the LLM, and the only place
  `create_notification` can be called

Make sure `Workshop2_Part2a_Server.ipynb` is running in another kernel before you run the
cells below.


## C1. Imports for the agent

In [2]:
import os
import asyncio
import logging
from pathlib import Path
from typing_extensions import TypedDict

from dotenv import load_dotenv
from transformers import pipeline
from langchain.chat_models import init_chat_model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END

load_dotenv()

BASE_DIR = Path(".").resolve()
ANNOUNCEMENTS_FILE = BASE_DIR / "workshop_outputs/announcements.txt"   # written by the create_notification tool

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


/opt/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## C2. Configure the LLM backend

Everything is set with plain variables in the next cell — no environment variables
needed (API keys still go in `.env`). Five presets are shown; uncomment the ONE you
want and re-run the cell.

**Tradeoff to know:** tool-calling reliability scales with model size/quality.
Qwen2.5-Instruct models support function calling even at 0.5B, but a 0.5B model will
miss or malform tool calls more often than a hosted API. If the agent seems to ignore
its tools (e.g. it never writes to `announcements.txt`), switch presets.

- **Local HF (default)** — runs anywhere (Mac or Gilbreth), no server or API key needed,
  but weakest tool-calling reliability.
- **Ollama** — free, and much better tool calling than the local default. Works out of
  the box on your Mac (`ollama serve` running, model pulled with `ollama pull
  llama3.2`). On Gilbreth it needs a one-time no-sudo userspace install (a static
  binary, no root required):
  ```bash
  curl -L https://ollama.com/download/ollama-linux-amd64.tgz -o ollama-linux-amd64.tgz
  mkdir -p ~/.local/bin
  tar -C ~/.local/bin -xzf ollama-linux-amd64.tgz
  echo 'export PATH="$HOME/.local/bin:$PATH"' >> ~/.bashrc && source ~/.bashrc

  # Home directory quota is usually small -- point model storage at scratch/depot instead:
  export OLLAMA_MODELS="/path/to/your/large/storage/directory"

  ollama serve            # keep this running in its own terminal for the whole session
  ollama pull llama3.2     # in a second terminal, once `ollama serve` is up
  ```
  Run this in a terminal within the same Gilbreth job/session as your Jupyter kernel,
  then use the Ollama preset below exactly as you would locally.
- **Purdue GenAI Studio** (`genai.rcac.purdue.edu`) — OpenAI-compatible endpoint, zero
  server management. Needs `pip install langchain-openai`. Put your GenAI Studio API key
  in `.env` as `OPENAI_API_KEY=...`. Verified working end-to-end (plain chat and tool
  calling both forwarded correctly by the endpoint).
- **Anthropic (Claude)** — put your key in `.env` as `ANTHROPIC_API_KEY`. Needs
  `pip install langchain-anthropic`. Very reliable tool calling.
- **OpenAI** — put your key in `.env` as `OPENAI_API_KEY`. Needs
  `pip install langchain-openai`. Very reliable tool calling. Don't combine with the
  Purdue GenAI preset above — both reuse the `OPENAI_API_KEY` name for different services.
</cell id="85919af7">


In [3]:
# --- Edit these directly, no env vars needed (API keys still go in .env). Uncomment ONE preset. ---

LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # used only when LLM_MODEL is None (the local preset)

# 1) Local HF model -- [local] runs anywhere (Mac or Gilbreth), no server or API key
#    needed, but weakest tool-calling reliability.
# LLM_MODEL    = None
# LLM_BASE_URL = None

# 2) Ollama -- [api, local server] free, much better tool calling than the local
#    default. Needs `ollama serve` running + `ollama pull llama3.2` (see the markdown
#    above for the one-time Gilbreth install steps).
LLM_MODEL    = "ollama:llama3.2"
LLM_BASE_URL = None

# 3) Purdue GenAI Studio (default) -- [api] OpenAI-compatible endpoint, zero server
#    management. Needs: pip install langchain-openai. Put your GenAI Studio API key in
#    .env as OPENAI_API_KEY=...
# LLM_MODEL    = "openai:llama3.1:latest"                    # or another model from your GenAI Studio account. adding openai prefix to the model name is required for GenAI Studio, because it is an OpenAI-compatible endpoint.  
# LLM_BASE_URL = "https://genai.rcac.purdue.edu/api"          # client appends /chat/completions itself

# 4) Anthropic (Claude) -- [api] very reliable tool calling. Needs:
#    pip install langchain-anthropic. Put your key in .env as ANTHROPIC_API_KEY.
# LLM_MODEL    = "anthropic:claude-opus-4-8"                 # or "anthropic:claude-sonnet-5" / "anthropic:claude-haiku-4-5" for cheaper/faster
# LLM_BASE_URL = None

# 5) OpenAI -- [api] very reliable tool calling. Needs: pip install langchain-openai.
#    Put your key in .env as OPENAI_API_KEY. Don't combine with the Purdue GenAI preset
#    above -- both reuse the OPENAI_API_KEY name for different services.
# LLM_MODEL    = "openai:gpt-4o-mini"                        # or "openai:gpt-4o" for a stronger model
# LLM_BASE_URL = None


def create_llm():
    """
    Set LLM_MODEL above to None for the local Hugging Face model (same as Workshop 1),
    or to a provider string to route through init_chat_model() instead -- Ollama,
    Purdue GenAI, Anthropic, OpenAI, or any OpenAI-compatible endpoint (LLM_BASE_URL
    lets you point at Open WebUI, vLLM, LM Studio, Purdue GenAI, etc.).
    """
    if LLM_MODEL is None:
        text_gen = pipeline("text-generation", model=LOCAL_MODEL, max_new_tokens=512)
        return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_gen))
    else:
        kwargs = {"temperature": 0}
        if LLM_BASE_URL:
            kwargs["base_url"] = LLM_BASE_URL
        return init_chat_model(LLM_MODEL, **kwargs)


print("LLM backend: ", f"local ({LOCAL_MODEL})" if LLM_MODEL is None else f"api ({LLM_MODEL})")


LLM backend:  api (ollama:llama3.2)


## C3. AgentState — the shared workflow state

Unlike a free-form ReAct loop where every node shares one growing message list, this
graph is a fixed pipeline: each node writes to its own field, so there's nothing to
merge and no reducer needed (compare to Workshop 1's `add_messages`, which had to append
to a shared list because multiple turns wrote to the same field).

In [4]:
class AgentState(TypedDict): # TypedDict for the agent's state, which is passed between nodes in the graph
    question:        str   # the user's question, set once at the start
    kb_result:       str   # search_knowledge_base output, filled in by search_kb_node
    calendar_result: str   # get_academic_calendar output, filled in by get_calendar_node
    final_answer:    str   # filled in by write_node once both lookups are done

## C4. Build the LangGraph agent graph

```
START ──── gather_context ──── write ──── END
```

If we use a generic loop where the LLM decided *whether* and *when* to call each tool,  the model could call `create_notification` before ever looking anything up. This graph fixes the order structurally instead, as two plain sequential steps:

- **`gather_context_node`** calls `search_knowledge_base` and `get_academic_calendar`
  directly (no LLM involved in deciding to run them -- they always run). The two calls
  don't depend on each other, so `asyncio.gather` runs them concurrently; this needs no
  special LangGraph feature, just plain Python.
- **`write_node`** runs after, with both results already sitting in state. It's the
  only place the LLM is called, and the only tool it's ever offered is
  `create_notification` -- there's no turn where writing a notification is possible
  before the facts exist.

No `ToolNode`, router, or tool-gating logic needed -- just two nodes in a straight line.


In [5]:
def build_graph(tools):
    llm = create_llm()
    tools_by_name = {t.name: t for t in tools}

    async def gather_context_node(state: AgentState) -> dict: # This node gathers context from the knowledge base and academic calendar tools in parallel, using asyncio.gather to run both lookups concurrently. The results are stored in the state dictionary for use in the next node.
        log.info("Node 1: gathering context...")
        kb_result, calendar_result = await asyncio.gather(
            tools_by_name["search_knowledge_base"].ainvoke({"query": state["question"]}),
            tools_by_name["get_academic_calendar"].ainvoke({"query": state["question"]}),
        )
        return {"kb_result": kb_result, "calendar_result": calendar_result}

    async def write_node(state: AgentState) -> dict:
        log.info("Node 2: writing the answer / notification...")
        # The retrieved context is already in the prompt below -- the model never has
        # to decide whether to look things up first, so it can't skip that step.
        notify_tool     = tools_by_name["create_notification"]
        llm_with_notify = llm.bind_tools([notify_tool])

        context = (
            f"Knowledge base results:\n{state['kb_result']}\n\n"
            f"Academic calendar results:\n{state['calendar_result']}"
        )
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=f"{state['question']}\n\nRetrieved context:\n{context}"),
        ]
        response = await llm_with_notify.ainvoke(messages)

        final_answer = response.content
        for tool_call in response.tool_calls:
            tool_result   = await notify_tool.ainvoke(tool_call["args"])
            final_answer += f"\n\n[create_notification] {tool_result}"

        return {"final_answer": final_answer}

    graph = StateGraph(AgentState) # The graph is built using the StateGraph class, which allows us to define nodes and edges that represent the flow of data and control between different parts of the agent's logic.
    graph.add_node("gather_context", gather_context_node) # Add the gather_context_node to the graph as a node named "gather_context"
    graph.add_node("write", write_node) # Add the write_node to the graph as a node named "write"
    graph.add_edge(START, "gather_context") # Define an edge from the START node to the "gather_context" node, indicating that the agent should start by gathering context
    graph.add_edge("gather_context", "write") # Define an edge from the "gather_context" node to the "write" node, indicating that after gathering context, the agent should proceed to write the answer
    graph.add_edge("write", END) # Define an edge from the "write" node to the END node, indicating that after writing the answer, the agent has completed its task
    return graph.compile()


## C5. System prompt for the Boilermaker TA

In [ ]:
SYSTEM_PROMPT = """
You are the Boilermaker Autonomous TA for a Purdue course.

You are given a question along with knowledge base and academic calendar results that
have already been retrieved for you -- you do not need to look anything up yourself.

Answer the question using only the retrieved context. If the question asks about an
existing fact (a policy, an exam date, a deadline) and the retrieved context doesn't
contain it, say so plainly instead of guessing.

Some questions instead ask you to schedule a NEW event (e.g. a review session) that
won't appear anywhere in the retrieved context, because it doesn't exist yet -- you have
to invent it. In that case, pick a specific date and time yourself, using the retrieved
academic calendar as constraints (avoid any date/time that conflicts with a listed
holiday, exam, or deadline), and state your choice plainly as your own proposal.

If the question asks for a notification or announcement to be written, call
create_notification with the subject and body filled in -- using retrieved facts where
they exist, or your own concrete proposed date/time for a new event you were asked to
schedule. Never use placeholder text like "[insert date]" or "[insert time]" -- always
commit to a real, specific value rather than leaving a gap.
"""

## C6. Connect to the MCP server and run the agent

In [7]:
DEFAULT_QUESTION = (
    "A professor wants to schedule a single CS course review session that does not conflict "
    "with holidays or exams. Summarize the plan and write a notification announcement for students."
)

MCP_SERVER_URL = "http://127.0.0.1:8001/mcp"


async def run_agent(question: str = DEFAULT_QUESTION):
    log.info("Connecting to MCP server at %s", MCP_SERVER_URL)

    try:
        client = MultiServerMCPClient({
            "boiler_ta": {
                "url": MCP_SERVER_URL,
                "transport": "streamable_http",
            },
        })
        tools = await client.get_tools()
    except Exception as e:
        log.error("Could not connect to MCP server: %s", e)
        return

    log.info("Loaded %d tools: %s", len(tools), [t.name for t in tools])

    agent  = build_graph(tools)
    result = await agent.ainvoke({"question": question})

    print("\nAgent reply:\n", result["final_answer"])


# Run the async agent inside the notebook
await run_agent()

13:17:29  INFO      Connecting to MCP server at http://127.0.0.1:8001/mcp
13:17:29  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:29  INFO      Received session ID: bb696bd5493f40e8a8602d16217fc638
13:17:29  INFO      Negotiated protocol version: 2025-11-25
13:17:29  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"
13:17:29  INFO      HTTP Request: GET http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:29  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:29  INFO      HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:29  INFO      Loaded 4 tools: ['search_knowledge_base', 'retrieve_docs', 'get_academic_calendar', 'create_notification']
13:17:29  INFO      Node 1: gathering context...
13:17:29  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:29  INFO      Received session ID: d777036badac48a6ab1c6b9df0ae2afa
13:17:29  INFO      Negotiated protoc


Agent reply:
 

[create_notification] [{'type': 'text', 'text': 'Notification written to /Users/ejebalba/Desktop/AI_Workshops/2ndWorkshop/workshop_outputs/announcements.txt.', 'id': 'lc_5d535586-42bd-433f-aa3c-9011ad37c4c8'}]


## C7. Try a custom question

In [8]:
await run_agent("What assignment deadlines are coming up in the next two weeks? Summarize them for students.")

13:17:36  INFO      Connecting to MCP server at http://127.0.0.1:8001/mcp
13:17:36  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:36  INFO      Received session ID: 7b92000b86aa4d078b67134be82ef873
13:17:36  INFO      Negotiated protocol version: 2025-11-25
13:17:36  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"
13:17:36  INFO      HTTP Request: GET http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:36  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:36  INFO      HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:36  INFO      Loaded 4 tools: ['search_knowledge_base', 'retrieve_docs', 'get_academic_calendar', 'create_notification']
13:17:36  INFO      Node 1: gathering context...
13:17:36  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:17:36  INFO      Received session ID: 46d9350efe824bc5a8ca88c0ad1499d8
13:17:36  INFO      Negotiated protoc


Agent reply:
 

[create_notification] [{'type': 'text', 'text': 'Notification written to /Users/ejebalba/Desktop/AI_Workshops/2ndWorkshop/workshop_outputs/announcements.txt.', 'id': 'lc_137bfb30-325b-4540-8afd-3527796f43a1'}]


## C8. Check the announcements file

In [9]:
if ANNOUNCEMENTS_FILE.exists():
    print(ANNOUNCEMENTS_FILE.read_text())
else:
    print("No announcements written yet.")

Subject: CS Course Review Session Schedule
A review session has been scheduled for the CS course to ensure students do not conflict with holidays or exams. The plan is as follows: The review session will be held on October 26th, immediately after the Fall Break. This ensures that students have had sufficient time to complete assignments and prepare for the midterm exams. The review session will cover topics covered in class and provide an opportunity for students to ask questions and clarify any doubts. We look forward to seeing you there!
---
Subject: Important: Upcoming Assignment Deadlines in CS 182
In the next two weeks, students can expect the following assignment deadlines:

* Midterm 2 for CS 182 on November 19th
* Midterm 1 for CS 182 on October 22nd

Please make sure to schedule study groups and plan accordingly.

Sincerely, [Your Name]
---



---
## Extension ideas

- Swap `LLM_MODEL` in `.env` to use a different backend (OpenAI, Purdue GenAI, vLLM)
- Add a new `@app.tool` in `Workshop2_Part2a_Server.ipynb` (e.g. `get_grades`,
  `send_email`) — since C4 wires specific tools by name into fixed graph nodes, using a
  new tool also means adding a node (and edges) for it in `build_graph`, not just
  restarting the server and re-running
- Point `MCP_SERVER_URL` at a server running on another machine or port
- Replace the FAISS index (in the server notebook) with a persistent vector database (ChromaDB, Qdrant)
- Connect the LoRA-fine-tuned model from Workshop 1 as the LLM backend
- Want to see the general ReAct-style version instead -- an `agent` node that decides
  which tools to call, plus a `ToolNode`, with the system prompt as the only thing
  enforcing order? Open **`Workshop2_Part2c_Agent_ReAct.ipynb`** -- it's an optional
  side-by-side comparison notebook built from exactly that pattern, including notes on
  how to test whether it still reproduces the placeholder-date bug this notebook fixed
